# Inferencia para Ollama

In [1]:
!curl -fsSL https://ollama.com/install.sh | sh

!pip install psutil

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import subprocess
import requests
import torch
import time
import re

### Configuración Global

In [3]:
print("Cargando dataset Dolly 15k...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
SAMPLE_SIZE = 75
test_subset = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE))

# --- PARÁMETROS ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPTS = [f"Instruction: {row['instruction']}\nResponse:" for row in test_subset]

# Configuración de generación idéntica para todos
GEN_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "top_k": 50,
    "do_sample": True
}

# Tabla para guardar resultados
results = []

# Determine if CUDA is available, otherwise use CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    print("CUDA está disponible. Usando GPU.")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32 # float16 is usually not optimized for CPU
    print("CUDA no está disponible. Usando CPU.")

print(f"Modelo: {MODEL_ID}")
print(f"Número de pruebas (Prompts): {len(PROMPTS)}")
print(f"Ejemplo de prompt:\n{PROMPTS[0][:100]}...")

Cargando dataset Dolly 15k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

CUDA está disponible. Usando GPU.
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Número de pruebas (Prompts): 75
Ejemplo de prompt:
Instruction: Who were the children of the legendary Garth Greenhand, the High King of the First Men ...


# OLLAMA

In [4]:
# Función para poder medir el uso de memoria, solo la usamos en OLLAMA
def get_ollama_memory_usage(model_name):
    """
    Analiza 'ollama ps' para devolver el uso de memoria desglosado.
    Se basa en el tamaño total y el % de GPU/CPU.
    """
    try:
        result = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
        if result.returncode != 0:
            return 0.0, 0.0

        lines = result.stdout.strip().split('\n')

        for line in lines[1:]:
            if model_name in line:
                # Busca patrones como "4.1 GB" o "400 MB"
                size_match = re.search(r'(\d+(?:\.\d+)?)\s*(GB|MB)', line)
                if not size_match:
                    continue

                val = float(size_match.group(1))
                unit = size_match.group(2)
                total_gb = val if unit == "GB" else val / 1024

                # Busca patrones como "100% GPU" o "40% GPU"
                gpu_match = re.search(r'(\d+)%\s*GPU', line)

                if gpu_match:
                    gpu_percent = float(gpu_match.group(1)) / 100.0
                else:
                    # Si no aparece "GPU", asumimos que está 100% en CPU
                    gpu_percent = 0.0

                vram_gb = total_gb * gpu_percent
                ram_gb = total_gb * (1.0 - gpu_percent)

                return vram_gb, ram_gb

        return 0.0, 0.0 # Modelo no encontrado
    except Exception as e:
        print(f"Error parseando memoria de Ollama: {e}")
        return 0.0, 0.0

print("--- Iniciando Benchmark: Ollama ---")

# Iniciar servidor
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# Descargar modelo
model_name = "tinyllama"
subprocess.run(["ollama", "pull", model_name], capture_output=True)

# Warmup
try:
    requests.post("http://localhost:11434/api/generate", json={
        "model": model_name,
        "prompt": "hi",
        "stream": False
    })
except Exception as e:
    print(f" Error en warmup: {e}")

# Medir RAM/VRAM
vram_gb, ram_gb = get_ollama_memory_usage(model_name)
total_mem_gb = vram_gb + ram_gb

# Loop
latencies = []
tokens_per_sec = []
url = "http://localhost:11434/api/generate"

for i, prompt in enumerate(tqdm(PROMPTS, desc="Ollama Requests")):
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": GEN_CONFIG["temperature"],
            "top_k": GEN_CONFIG["top_k"],
            "num_predict": GEN_CONFIG["max_new_tokens"]
        }
    }

    try:
        start_wall = time.time()
        response = requests.post(url, json=payload)
        end_wall = time.time()

        if response.status_code == 200:
            data = response.json()
            eval_count = data.get('eval_count', 0)
            eval_dur_ns = data.get('eval_duration', 0)

            if eval_dur_ns > 0:
                eval_dur_s = eval_dur_ns / 1e9
                tps = eval_count / eval_dur_s
                tokens_per_sec.append(tps)
                latencies.append(end_wall - start_wall)
        else:
            print(f"Error HTTP {response.status_code}")

    except Exception as e:
        print(f"Excepción en prompt {i}: {e}")

# Guardar resultados
if tokens_per_sec:
    avg_tps = np.mean(tokens_per_sec)
    avg_lat = np.mean(latencies)

    results.append({
        "Framework": "Ollama",
        "Avg Latency (s)": avg_lat,
        "Throughput (tok/s)": avg_tps,
        "VRAM Usage (GB)": vram_gb,
        "RAM Usage (GB)": ram_gb,
        "Total Memory (GB)": total_mem_gb
    })

else:
    print(" No se obtuvieron métricas válidas.")

ollama_process.terminate()

--- Iniciando Benchmark: Ollama ---


Ollama Requests: 100%|██████████| 75/75 [00:43<00:00,  1.73it/s]


In [5]:
results

[{'Framework': 'Ollama',
  'Avg Latency (s)': np.float64(0.5777874533335368),
  'Throughput (tok/s)': np.float64(255.12455691610788),
  'VRAM Usage (GB)': 0.78125,
  'RAM Usage (GB)': 0.0,
  'Total Memory (GB)': 0.78125}]